#### Interact with the website
#### Download files

In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
url="https://ocrportal.hhs.gov/ocr/breach/breach_report_hip.jsf"
response=requests.get(url)

In [ ]:
print(BeautifulSoup(response.text).prettify())

In [ ]:
from playwright.async_api import async_playwright

In [ ]:
p = await async_playwright().start()
browser = await p.chromium.launch(headless=False, slow_mo=500)
page = await browser.new_page()

await page.goto(url)

await page.get_by_role("link", name="Archive", exact=True).click()
#download the csv in the current working directory
async with page.expect_download() as download_info:
    await page.get_by_role("link", name="CSV").click()

download = await download_info.value
await download.save_as("archive.csv")

await browser.close()
await p.stop()

#### Download the forms

In [ ]:
# url="https://extapps2.oge.gov/201/Presiden.nsf/API.xsp/v3/rest

In [ ]:
url="https://extapps2.oge.gov/201/Presiden.nsf/API.xsp/rest?"
   
response = requests.get(url)
data=response.json()
print(data.keys())

import pandas as pd
df=pd.DataFrame(data['data'])

df.shape


In [ ]:
df.head()

In [ ]:
for i in range(0, data['recordsTotal'], 100):
    url=f"https://extapps2.oge.gov/201/Presiden.nsf/API.xsp/v3/rest?start={i}"
    response=requests.get(url)
    content=response.json()
    data=content['data']
    df=pd.DataFrame(data)
    # print(df.shape)
    if 'all_data' not in locals():
        all_data=df
    else:
        all_data=pd.concat([all_data, df], ignore_index=True)

all_data.shape

In [ ]:
df=all_data.copy()

In [ ]:
## Convert docDate to datetime
df['date_added']=pd.to_datetime(df['docDate'])

df.date_added.min(), df.date_added.max()

In [ ]:
df['type'][60]

In [ ]:
## extraction using pattern
df['type'].str.extract(r"href=['\"](.*?)['\"]", expand=False)[0]

In [ ]:
# why not use bs4 to parse it. it is cleaner and more accurate
def get_link(text):
    a = BeautifulSoup(str(text), "html.parser").find("a")
    if a is None:
        return pd.Series([None, None])
    return pd.Series([a.get("href"), a.get_text(strip=True)])

df[['link', 'link_text']] = df['type'].apply(get_link)

In [ ]:
df['type'].str.contains("annual", case=False, na=False).sum()

In [ ]:
# Make sure output folder exists
import os
os.makedirs("./forms/", exist_ok=True)

# Loop through DataFrame rows
for i, row in df.head(5).iterrows():
    url = row['link']
    date = pd.to_datetime(row['date_added']).strftime('%Y%m%d')

    # Clean and format the base filename
    base_name = row['name'].replace(" ", "-").replace(",", "")
    extension = ".pdf"

    # Include the date in the filename immediately
    output_name = f"{base_name}_{date}{extension}"
    output_path = f"./forms/{output_name}"

    try:
        response = requests.get(url)
        response.raise_for_status()

        with open(output_path, "wb") as f:
            f.write(response.content)

        print(f"Downloaded: {output_path}")

    except Exception as e:
        print(f"Failed to download {url}: {e}")